In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!nvidia-smi


Fri Dec 12 02:17:59 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             12W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pandas as pd

# Upload the CSV file to Colab first (via Files tab or code below)
from google.colab import files
uploaded = files.upload()
path = list(uploaded.keys())[0]  # Gets the uploaded file name dynamically

df = pd.read_csv(path)
df.head()
df.shape
df.columns
df.isnull().sum()


KeyboardInterrupt: 

In [ ]:
data = df[['text_combined', 'label']].copy()
data.rename(columns={'text_combined': 'text'}, inplace=True)
data.head()


In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'<.*?>', ' ', t)
    t = re.sub(r'[^a-zA-Z]', ' ', t)
    t = ' '.join(w for w in t.split() if w not in stop_words)
    return t

data['clean_text'] = data['text'].apply(clean_text)
data.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'], data['label'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

model = LinearSVC()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
from transformers import BertTokenizer
import tensorflow as tf


!pip install transformers -q

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_batch(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize_batch(X_train)
test_encodings = tokenize_batch(X_test)


In [ ]:
!pip install transformers torch -q

import torch
from transformers import BertTokenizer, BertForSequenceClassification
import numpy as np
from sklearn.model_selection import train_test_split

# Reuse your existing tokenizer/encodings
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenize_batch(X_train)
test_encodings = tokenize_batch(X_test)

print("PyTorch BERT ready!")


In [ ]:
import torch.nn as nn

class BertLSTMClassifier(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased', num_classes=1, lstm_hidden=128):
        super().__init__()
        self.bert = BertForSequenceClassification.from_pretrained(bert_model_name, num_labels=num_classes)
        self.lstm = nn.LSTM(768, lstm_hidden, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(lstm_hidden*2, num_classes)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.bert.bert(input_ids, attention_mask=attention_mask)
        lstm_out, (hn, cn) = self.lstm(outputs.last_hidden_state)
        pooled = hn[-2:].transpose(0,1).contiguous().view(hn.size(1), -1)
        dropped = self.dropout(pooled)
        logits = self.classifier(dropped)
        return self.sigmoid(logits)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertLSTMClassifier().to(device)
print(f"Model on: {device}")


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Convert encodings and labels to tensors
train_input_ids = torch.tensor(train_encodings['input_ids'])
train_attention_mask = torch.tensor(train_encodings['attention_mask'])
train_labels = torch.tensor(y_train.values).long()

test_input_ids = torch.tensor(test_encodings['input_ids'])
test_attention_mask = torch.tensor(test_encodings['attention_mask'])
test_labels = torch.tensor(y_test.values).long()

# Create TensorDatasets
train_dataset = TensorDataset(train_input_ids, train_attention_mask, train_labels)
test_dataset = TensorDataset(test_input_ids, test_attention_mask, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print(f" Batch=16 Ready: Train={len(train_loader)} batches")
print(f"GPU Free: ~{14 - torch.cuda.memory_reserved()/1e9:.1f}GB")

In [ ]:
from torch.optim import Adam
import torch.nn as nn

optimizer = Adam(model.parameters(), lr=2e-5)
criterion = nn.BCELoss()
device = next(model.parameters()).device

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, batch in enumerate(loader):
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs.squeeze(), labels.float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        predicted = (outputs > 0.5).float()
        total += labels.size(0)
        correct += (predicted.squeeze() == labels).sum().item()

        del input_ids, attention_mask, labels, outputs
        torch.cuda.empty_cache()

    return total_loss/len(loader), correct/total

print("TRAINING STARTED (2 epochs, batch=16)...")
for epoch in range(2):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Epoch {epoch+1}/2: Loss={train_loss:.4f}, Acc={train_acc:.4f}")
    torch.cuda.empty_cache()


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)

            outputs = model(input_ids, attention_mask)
            predicted = (outputs > 0.5).float()

            total += labels.size(0)
            correct += (predicted.squeeze() == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            del input_ids, attention_mask, labels, outputs
            torch.cuda.empty_cache()

    from sklearn.metrics import classification_report
    print(f" FINAL Test Accuracy: {correct/total:.4f}")
    print("\n Detailed Report:")
    print(classification_report(all_labels, all_preds))
    return correct/total

test_acc = evaluate(model, test_loader, device)


In [ ]:
def predict_email(text, model, tokenizer, device):
    model.eval()
    cleaned = clean_text(text)
    encoding = tokenizer(cleaned, return_tensors='pt',
                        padding=True, truncation=True, max_length=128)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        output = model(input_ids, attention_mask)
        prob = output.squeeze().cpu().numpy()

    return "PHISHING" if prob > 0.5 else "LEGITIMATE", prob

# Test it!
test_email = "Click here to claim your $1000 prize! Urgent action required."
result, confidence = predict_email(test_email, model, tokenizer, device)
print(f"Email: {test_email}")
print(f"Prediction: {result} (confidence: {confidence:.3f})")
